# Factor Graph Trajectory Solving From Simulated Rover Data

This notebook generates the same planar rover simulation style used in notebook 04, builds rolling MROB calibration graphs through `FactorGraphCalibration`, and compares the true simulated trajectory with the trajectory recovered from IMU and LiDAR observations.

## 1. Configuration

All tunable settings live here so the later simulated-data import block can be replaced by real data without hunting through execution cells.

In [1]:
# Trajectory controls matching notebook 04 defaults.
RECTANGLE_COUNT = 2
RECTANGLE_WIDTH = 10.0
RECTANGLE_HEIGHT = 5.0
STRAIGHT_SPEED = 1.0
TURN_DURATION = 1.5

# Sensor controls.
IMU_RATE_HZ = 100.0
LIDAR_RATE_HZ = 5.0
RANDOM_SEED = 52
FIXED_EXTRINSIC = "T_B_L"

# Initial-guess controls. Endpoints are kept exact because they are anchored.
INITIAL_POSE_ROT_STD_RAD = 0.015
INITIAL_POSE_XY_STD_M = 0.08
INITIAL_POSE_Z_STD_M = 0.0
INITIAL_POSE_RANDOM_SEED = 7

# Rolling factor-graph controls.
WINDOW_SIZE = 5.0
STEP_SIZE = 1.0
IMU_SAMPLES_PER_FACTOR = 100
LIDAR_SAMPLES_PER_FACTOR = 20
FACTOR_GRAPH_METHOD = "LM"

# Output controls.
SAVE_FIGURES = True

## 2. Imports

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "calib_observability").exists():
        ROOT = candidate
        break
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

OUT = ROOT / "outputs" / "factor_graph_simulated_trajectory"
OUT.mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt

from calib_observability.lie_se3 import se3_exp
from calib_observability.plotting import plot_calibration_window_chi2, plot_trajectory_comparison
from calib_observability.simulation import PlanarRoverConfig, reframe_dataset_to_fixed_extrinsic, simulate_planar_rover
from factor_graph_calibration import FactorGraphCalibration

## 3. Data Import: Simulated Rover Observations

This cell is intentionally the only place that generates/imports observations. To switch to a real dataset later, replace the assignments produced here: `pose_timestamps`, `initial_poses`, `true_poses`, `imu_timestamps`, `gyrs`, `accs`, `lidar_timestamps`, `lidar_poses`, and calibration initials.

In [3]:
rover_mode = "multiple_rectangles" if RECTANGLE_COUNT > 1 else "one_rectangle"
rover_config = PlanarRoverConfig(
    rectangle_width=RECTANGLE_WIDTH,
    rectangle_height=RECTANGLE_HEIGHT,
    straight_speed=STRAIGHT_SPEED,
    turn_duration=TURN_DURATION,
    total_laps=RECTANGLE_COUNT,
    imu_rate_hz=IMU_RATE_HZ,
    lidar_rate_hz=LIDAR_RATE_HZ,
    random_seed=RANDOM_SEED,
    mode=rover_mode,
)

raw_dataset = simulate_planar_rover(rover_config, mode=rover_mode)
dataset = reframe_dataset_to_fixed_extrinsic(raw_dataset, FIXED_EXTRINSIC)

pose_timestamps = dataset.lidar.true_times.copy()
true_poses = np.stack([dataset.trajectory.pose_at(float(t)) for t in pose_timestamps], axis=0)

imu_timestamps = dataset.imu.sensor_timestamps.copy()
gyrs = dataset.imu.gyroscope.copy()
accs = dataset.imu.accelerometer.copy()

lidar_timestamps = dataset.lidar.sensor_timestamps.copy()
lidar_poses = np.empty((lidar_timestamps.size, 4, 4), dtype=float)
lidar_poses[0] = np.eye(4)
for measurement_index, relative_pose in enumerate(dataset.lidar.measurements):
    lidar_poses[measurement_index + 1] = lidar_poses[measurement_index] @ relative_pose

# Build a slightly perturbed trajectory initial guess while preserving anchored endpoints.
rng = np.random.default_rng(INITIAL_POSE_RANDOM_SEED)
pose_perturbations = np.zeros((pose_timestamps.size, 6), dtype=float)
pose_perturbations[:, :3] = rng.normal(0.0, INITIAL_POSE_ROT_STD_RAD, size=(pose_timestamps.size, 3))
pose_perturbations[:, 3] = rng.normal(0.0, INITIAL_POSE_XY_STD_M, size=pose_timestamps.size)
pose_perturbations[:, 4] = rng.normal(0.0, INITIAL_POSE_XY_STD_M, size=pose_timestamps.size)
pose_perturbations[:, 5] = rng.normal(0.0, INITIAL_POSE_Z_STD_M, size=pose_timestamps.size)
pose_perturbations[0] = 0.0
pose_perturbations[-1] = 0.0
initial_poses = np.stack([se3_exp(xi) @ pose for xi, pose in zip(pose_perturbations, true_poses)], axis=0)

T_B_I_initial = dataset.T_B_I_true.copy()

print(f"trajectory mode: {dataset.trajectory.mode}")
print(f"fixed extrinsic: {FIXED_EXTRINSIC}, T_B_L identity: {np.allclose(dataset.T_B_L_true, np.eye(4))}")
print(f"pose nodes: {pose_timestamps.size}")
print(f"IMU samples: {imu_timestamps.size}, LiDAR odometry poses: {lidar_poses.shape[0]}")
print(f"true tau_I={dataset.tau_I_true:+.3f}s, true tau_L={dataset.tau_L_true:+.3f}s")

trajectory mode: multiple_rectangles_B_equals_L
fixed extrinsic: T_B_L, T_B_L identity: True
pose nodes: 361
IMU samples: 7201, LiDAR odometry poses: 361
true tau_I=+0.010s, true tau_L=-0.020s


## 4. Run Rolling Factor Graph

This cell is only the algorithm execution. The call uses `FactorGraphCalibration.generate_filter_iterative` from `src/factor_graph_calibration.py`.

In [4]:
fgraph = FactorGraphCalibration(
    imu_samples_per_factor=IMU_SAMPLES_PER_FACTOR,
    lidar_samples_per_factor=LIDAR_SAMPLES_PER_FACTOR,
    T_B_I_anchor=True,
    T_B_L_anchor=True,
    bias_anchor=True,
    tau_I_anchor=True,
    tau_L_anchor=True,
    anchor_first_pose=True,
    anchor_last_pose=True,
    method=FACTOR_GRAPH_METHOD,
    imu_time_offset_margin=2.5,
    lidar_time_offset_margin=2.5,
    tau_I_regularization_information=100,
    scheduler=[(1e-5, 1)],
    solver_verbose=True,
    include_lidar_factors=False,
)

reset: releasing old C++ graph
reset: old C++ graph released
reset: new C++ graph created


In [5]:
result = fgraph.generate_filter_window(
    window_index=0,
    window_start=2.5,
    window_end=0 + 10,
    pose_timestamps=pose_timestamps,
    initial_poses=initial_poses,
    imu_timestamps=imu_timestamps,
    angular_velocity_imu=gyrs,
    specific_force_imu=accs,
    # lidar_timestamps=lidar_timestamps,
    # lidar_odometry_poses=lidar_poses,
    T_B_I_initial=T_B_I_initial,
    verbose=2,
)

reset: releasing old C++ graph
Old filter object is not none <mrob.pybind.FGraph object at 0x7fdda29e8cf0>
reset: old C++ graph released
reset: new C++ graph created
WINDOW 0, RAW [2.5, 10]:
Chi2 error = 7313.481466046356
Nodes = 41, factors = 75
Factor counts = {'gyro': 37, 'accel': 38, 'lidar': 0, 'bias_prior': 0, 'tau_I_prior': 0, 'tau_L_prior': 0, 'T_B_I_prior': 0, 'T_B_L_prior': 0}
pose[0] at t=2.6:
[[ 9.99159963e-01 -4.09466083e-02  1.65615885e-03  2.89456526e+00]
 [ 4.09438330e-02  9.99160043e-01  1.67632007e-03  1.86219748e-01]
 [-1.72340737e-03 -1.60710241e-03  9.99997224e-01  1.14955018e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
pose[1] at t=2.8000000000000003:
[[ 9.98176520e-01 -6.03549735e-02  9.54700244e-04  3.31122478e+00]
 [ 6.03272794e-02  9.98009321e-01  1.83851656e-02  4.24336101e-02]
 [-2.06243593e-03 -1.82940462e-02  9.99830523e-01  1.15402439e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
pose[2] at t=3.0:
[[ 

FGraph::~FGraph begin
  factors = 0
  nodes = 0
  active nodes = 0
  eigen factors = 0
FGraph::~FGraph complete


In [6]:
print("FIRST WINDOW COMPLETE", flush=True)

fgraph.reset(clear_rolling_state=False)

print("RESET COMPLETE", flush=True)

FIRST WINDOW COMPLETE


: 

In [ ]:
import gc
import mrob
import numpy as np

for iteration in range(1000):
    graph = mrob.FGraph()

    origin = graph.add_node_pose_3d(
        mrob.SE3(),
        mrob.NODE_ANCHOR,
    )
    target = graph.add_node_pose_3d(
        mrob.SE3(),
        mrob.NODE_ANCHOR,
    )
    extrinsic = graph.add_node_pose_3d(
        mrob.SE3(),
        mrob.NODE_ANCHOR,
    )
    bias = graph.add_node_landmark_3d(
        np.zeros(3),
        mrob.NODE_ANCHOR,
    )
    tau = graph.add_node_scalar(
        0.0,
        mrob.NODE_ANCHOR,
    )

    timestamps = np.linspace(-1.0, 2.0, 100)
    measurements = np.zeros((len(timestamps), 3))

    graph.add_factor_(
        0.0,
        1.0,
        timestamps,
        measurements,
        origin,
        target,
        extrinsic,
        bias,
        tau,
        np.eye(3),
    )

    graph.chi2()

    del graph
    gc.collect()

    print("destroyed", iteration, flush=True)

destroyed 0
destroyed 1
destroyed 2


FGraph::~FGraph begin
  factors = 1
  nodes = 5
  active nodes = 0
  eigen factors = 0
Destroying factor
  id = 0
  dynamic type = N4mrob19FactorGyroCalibPropE
  remaining = 0
~FactorGyroCalibProp begin, id = 0
Clearing angular_velocity_imu
angular_velocity_imu cleared
Clearing sensor_timestamps
sensor_timestamps cleared
~FactorGyroCalibProp body complete
Factor destroyed successfully
Destroying node
  id = 4
  dynamic type = N4mrob10NodeScalarE
  remaining = 4
Node destroyed successfully
Destroying node
  id = 3
  dynamic type = N4mrob14NodeLandmark3dE
  remaining = 3
Node destroyed successfully
Destroying node
  id = 2
  dynamic type = N4mrob10NodePose3dE
  remaining = 2
Node destroyed successfully
Destroying node
  id = 1
  dynamic type = N4mrob10NodePose3dE
  remaining = 1
Node destroyed successfully
Destroying node
  id = 0
  dynamic type = N4mrob10NodePose3dE
  remaining = 0
Node destroyed successfully
FGraph::~FGraph complete
FGraph::~FGraph begin
  factors = 1
  nodes = 5
  act

destroyed 3
destroyed 4
destroyed 5
destroyed 6


FGraph::~FGraph begin
  factors = 1
  nodes = 5
  active nodes = 0
  eigen factors = 0
Destroying factor
  id = 0
  dynamic type = N4mrob19FactorGyroCalibPropE
  remaining = 0
~FactorGyroCalibProp begin, id = 0
Clearing angular_velocity_imu
angular_velocity_imu cleared
Clearing sensor_timestamps
sensor_timestamps cleared
~FactorGyroCalibProp body complete
Factor destroyed successfully
Destroying node
  id = 4
  dynamic type = N4mrob10NodeScalarE
  remaining = 4
Node destroyed successfully
Destroying node
  id = 3
  dynamic type = N4mrob14NodeLandmark3dE
  remaining = 3
Node destroyed successfully
Destroying node
  id = 2
  dynamic type = N4mrob10NodePose3dE
  remaining = 2
Node destroyed successfully
Destroying node
  id = 1
  dynamic type = N4mrob10NodePose3dE
  remaining = 1
Node destroyed successfully
Destroying node
  id = 0
  dynamic type = N4mrob10NodePose3dE
  remaining = 0
Node destroyed successfully
FGraph::~FGraph complete
FGraph::~FGraph begin
  factors = 1
  nodes = 5
  act

destroyed 7
destroyed 8
destroyed 9
destroyed 10
destroyed 11


...

destroyed 142
destroyed 143
destroyed 144
destroyed 145
destroyed 146


FGraph::~FGraph begin
  factors = 1
  nodes = 5
  active nodes = 0
  eigen factors = 0
Destroying factor
  id = 0
  dynamic type = N4mrob19FactorGyroCalibPropE
  remaining = 0
~FactorGyroCalibProp begin, id = 0
Clearing angular_velocity_imu
angular_velocity_imu cleared
Clearing sensor_timestamps
sensor_timestamps cleared
~FactorGyroCalibProp body complete
Factor destroyed successfully
Destroying node
  id = 4
  dynamic type = N4mrob10NodeScalarE
  remaining = 4
Node destroyed successfully
Destroying node
  id = 3
  dynamic type = N4mrob14NodeLandmark3dE
  remaining = 3
Node destroyed successfully
Destroying node
  id = 2
  dynamic type = N4mrob10NodePose3dE
  remaining = 2
Node destroyed successfully
Destroying node
  id = 1
  dynamic type = N4mrob10NodePose3dE
  remaining = 1
Node destroyed successfully
Destroying node
  id = 0
  dynamic type = N4mrob10NodePose3dE
  remaining = 0
Node destroyed successfully
FGraph::~FGraph complete
FGraph::~FGraph begin
  factors = 1
  nodes = 5
  act

destroyed 147
destroyed 148


FGraph::~FGraph begin
  factors = 1
  nodes = 5
  active nodes = 0
  eigen factors = 0
Destroying factor
  id = 0
  dynamic type = N4mrob19FactorGyroCalibPropE
  remaining = 0
~FactorGyroCalibProp begin, id = 0
Clearing angular_velocity_imu
angular_velocity_imu cleared
Clearing sensor_timestamps
sensor_timestamps cleared
~FactorGyroCalibProp body complete
Factor destroyed successfully
Destroying node
  id = 4
  dynamic type = N4mrob10NodeScalarE
  remaining = 4
Node destroyed successfully
Destroying node
  id = 3
  dynamic type = N4mrob14NodeLandmark3dE
  remaining = 3
Node destroyed successfully
Destroying node
  id = 2
  dynamic type = N4mrob10NodePose3dE
  remaining = 2
Node destroyed successfully
Destroying node
  id = 1
  dynamic type = N4mrob10NodePose3dE
  remaining = 1
Node destroyed successfully
Destroying node
  id = 0
  dynamic type = N4mrob10NodePose3dE
  remaining = 0
Node destroyed successfully
FGraph::~FGraph complete
FGraph::~FGraph begin
  factors = 1
  nodes = 5
  act

KeyboardInterrupt: 

In [5]:
filter_graph = FactorGraphCalibration(
    imu_samples_per_factor=IMU_SAMPLES_PER_FACTOR,
    lidar_samples_per_factor=LIDAR_SAMPLES_PER_FACTOR,
    T_B_I_anchor=True,
    T_B_L_anchor=True,
    bias_anchor=True,
    tau_I_anchor=True,
    tau_L_anchor=True,
    anchor_first_pose=True,
    anchor_last_pose=True,
    method=FACTOR_GRAPH_METHOD,
    imu_time_offset_margin=2.5,
    lidar_time_offset_margin=2.5,
    tau_I_regularization_information=100,
    scheduler=[(1e-5, 1)],
    solver_verbose=True,
)

# filter_graph.add_tau_regularization_factor(filter_graph.node_tau_I, target=0, information=10000000)

results = filter_graph.generate_filter_iterative(
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    pose_timestamps=pose_timestamps,
    initial_poses=initial_poses,
    imu_timestamps=imu_timestamps,
    angular_velocity_imu=gyrs,
    specific_force_imu=accs,
    lidar_timestamps=lidar_timestamps,
    lidar_odometry_poses=lidar_poses,
    T_B_I_initial=T_B_I_initial,
    T_B_L_initial=np.eye(4),
    bias_initial=np.zeros(3),
    tau_I_initial=0.0,
    tau_L_initial=0.0,
    verbose=1,
)

estimated_timestamps, estimated_poses = filter_graph.rolling_trajectory

print(f"solved windows: {len(results)}")
print(f"stitched estimated poses: {estimated_poses.shape}")
if results:
    print(f"first chi2: {results[0].chi2_before:.3e} -> {results[0].chi2_after:.3e}")
    print(f"last chi2:  {results[-1].chi2_before:.3e} -> {results[-1].chi2_after:.3e}")
    print(f"last tau_I estimate: {results[-1].tau_I}")
    print(f"last bias estimate: {results[-1].bias_g}")

: 

In [1]:
import mrob
import numpy as np

graph = mrob.FGraph()

for i in range(1000):
    tau = graph.add_node_scalar(
        0.2,
        mrob.NODE_STANDARD,
    )

    graph.add_factor_1_scalar_obs(
        0.0,
        tau,
        np.array([[100.0]]),
    )

print("state before:", graph.get_estimated_state()[tau], flush=True)
print("chi2 before:", graph.chi2(), flush=True)

graph.solve(
    method=mrob.LM,
    maxIters=5,
    lambdaParam=1e-2,
    solutionTolerance=1e-12,
    verbose=True,
)

print("state after:", graph.get_estimated_state()[tau], flush=True)
print("chi2 after:", graph.chi2(), flush=True)

state before: [[0.2]]
chi2 before: 2000.0
state after: [[3.12458989e-15]]

FGraphSolve::optimize_levenberg_marquardt: iteration 1 lambda = 0.01, error 2000, and delta = 2000
model fidelity = 1.0102 and m_k = 1979.8

FGraphSolve::optimize_levenberg_marquardt: iteration 2 lambda = 0.0025, error 1.9996e-05, and delta = 1.9996e-05
model fidelity = 1.00253 and m_k = 1.99455e-05

FGraphSolve::optimize_levenberg_marquardt: iteration 3 lambda = 0.000625, error 1.24969e-14, and delta = 1.24969e-14

FGraphSolve::optimize_levenberg_marquardt: Converged Successfully

Time profile for 2.118 [ms]: 
Gauss Newton solve Cholesky = 0.566572%,
Gauss Newton create Cholesky = 21.6242%,
Info Adjacency = 33.6166%,
Adjacency = 44.1926%,

chi2 after: 4.881530981679374e-25


## 5. Visualize True, Initial, And Calculated Trajectories

In [ ]:
fig, axes = plot_trajectory_comparison(
    true_poses,
    estimated_poses,
    true_timestamps=pose_timestamps,
    estimated_timestamps=estimated_timestamps,
    initial_poses=initial_poses,
    title="Simulated true vs factor-graph trajectory",
)
if SAVE_FIGURES:
    fig.savefig(OUT / "trajectory_comparison.png", dpi=160)
plt.show()
plt.close(fig)

In [ ]:
fig, axis = plot_calibration_window_chi2(results)
if SAVE_FIGURES:
    fig.savefig(OUT / "rolling_window_chi2.png", dpi=160)
plt.show()
plt.close(fig)

## 6. Window Summary Table

In [ ]:
summary_rows = []
for result in results:
    summary_rows.append({
        "window": result.window_index,
        "start": result.window_start,
        "end": result.window_end,
        "poses": result.pose_timestamps.size,
        "chi2_before": result.chi2_before,
        "chi2_after": result.chi2_after,
        "gyro_factors": result.factor_counts.get("gyro", 0),
        "accel_factors": result.factor_counts.get("accel", 0),
        "lidar_factors": result.factor_counts.get("lidar", 0),
        "tau_I": result.tau_I,
        "tau_L": result.tau_L,
    })

try:
    import pandas as pd
    display(pd.DataFrame(summary_rows))
except ImportError:
    summary_rows[:5]